In [1]:
import pandas as pd
import numpy as np

# Path file
file_2027 = "template-erdkk-2027 - Sumber Makmur (1).xlsx"
file_pencocokan = "Copy of Hasil_Pencocokan_Petani.xlsx"
file_dosis = "Dosis pemupukan kecamatan Kutasari.xlsx"

# Baca data
df_2027 = pd.read_excel(file_2027)
df_pencocokan = pd.read_excel(file_pencocokan)
df_dosis = pd.read_excel(file_dosis)

print("Semua file berhasil dibaca!")

Semua file berhasil dibaca!


In [2]:
def clean_nik(val):
    if pd.isna(val):
        return ""
    val_str = str(val).strip().replace("`", "").replace("'", "")
    if val_str.endswith(".0"):
        val_str = val_str[:-2]
    return val_str

# Kumpulkan NIK yang sudah ada di template 2027
df_2027['NIK_CLEAN'] = df_2027['KTP'].apply(clean_nik)
existing_niks = set(df_2027['NIK_CLEAN'])
existing_niks.discard("")

print(f"Jumlah baris awal di template: {len(df_2027)}")

Jumlah baris awal di template: 55


In [3]:
df_pencocokan['Nama Poktan Normal'] = df_pencocokan['Nama Poktan'].astype(str).str.strip().str.upper()
df_sumber_makmur = df_pencocokan[df_pencocokan['Nama Poktan Normal'] == 'SUMBER MAKMUR'].copy()
df_sumber_makmur['NIK_CLEAN'] = df_sumber_makmur['KTP'].apply(clean_nik)

# Ambil yang belum ada di template
df_new = df_sumber_makmur[~df_sumber_makmur['NIK_CLEAN'].isin(existing_niks)].copy()
df_new = df_new.drop_duplicates(subset=['NIK_CLEAN'])

print(f"Data baru Sumber Makmur yang akan ditambahkan: {len(df_new)}")

Data baru Sumber Makmur yang akan ditambahkan: 36


In [4]:
column_mapping = {
    'Nama Penyuluh': 'Nama Penyuluh',
    'NIK_CLEAN': 'KTP',
    'Nama Ibu Kandung': 'Nama Ibu Kandung',
    'Kode Desa': 'Kode Desa',
    'Kode Kios Pengecer': 'Kode Kios Pengecer',
    'Subsektor': 'Subsektor',
    
    # Periode Tanam I
    'Komoditas MT1': 'Komoditas Periode Tanam I',
    'Luas Lahan (Ha) MT1': 'Luas Lahan (Ha) Periode Tanam I',
    
    # Periode Tanam II
    'Komoditas MT2': 'Komoditas Periode Tanam II',
    'Luas Lahan (Ha) MT2': 'Luas Lahan (Ha) Periode Tanam II',
    
    # Periode Tanam III
    'Komoditas MT3': 'Komoditas Periode Tanam III',
    'Luas Lahan (Ha) MT3': 'Luas Lahan (Ha) Periode Tanam III'
}

df_to_append = pd.DataFrame()
for col_copy, col_temp in column_mapping.items():
    if col_copy in df_new.columns:
        df_to_append[col_temp] = df_new[col_copy]
    else:
        df_to_append[col_temp] = None

df_to_append['ID Poktan'] = 537293

template_cols = [c for c in df_2027.columns if c != 'NIK_CLEAN']
df_to_append = df_to_append.reindex(columns=template_cols)
df_template_clean = df_2027[template_cols].copy()

# Gabung data
df_final = pd.concat([df_template_clean, df_to_append], ignore_index=True)

print(f"Total baris gabungan sementara: {len(df_final)}")

Total baris gabungan sementara: 91


In [5]:
# 1. Siapkan data dosis
dosis_dict = {}
for _, r in df_dosis.iterrows():
    k = str(r['komoditas']).strip().upper()
    dosis_dict[k] = {
        'urea': float(r['urea']) if pd.notna(r['urea']) else 0.0,
        'npk': float(r['npk']) if pd.notna(r['npk']) else 0.0,
        'npk_formula': float(r['npk formula']) if pd.notna(r['npk formula']) else 0.0,
        'organik': float(r['organik']) if pd.notna(r['organik']) else 0.0,
        'za': float(r['za']) if pd.notna(r['za']) else 0.0
    }

# 2. Hitung pupuk untuk setiap periode tanam
periodes = [
    ('I', 'Komoditas Periode Tanam I', 'Luas Lahan (Ha) Periode Tanam I'),
    ('II', 'Komoditas Periode Tanam II', 'Luas Lahan (Ha) Periode Tanam II'),
    ('III', 'Komoditas Periode Tanam III', 'Luas Lahan (Ha) Periode Tanam III')
]

for p_num, kom_col, luas_col in periodes:
    col_urea = f'Pupuk Urea (Kg) Periode Tanam {p_num}'
    col_npk = f'Pupuk NPK (Kg) Periode Tanam {p_num}'
    col_npk_form = f'Pupuk NPK Formula Khusus (Kg) Periode Tanam {p_num}'
    col_organik = f'Pupuk Organik (Kg) Periode Tanam {p_num}'
    col_za = f'Pupuk Za (Kg) Periode Tanam {p_num}'
    
    def hitung_pupuk(row, jenis):
        komoditas = str(row[kom_col]).strip().upper() if pd.notna(row[kom_col]) else ""
        luas = row[luas_col]
        
        if komoditas in dosis_dict and pd.notna(luas):
            try:
                luas_val = float(luas)
                if luas_val > 0:
                    return round(luas_val * dosis_dict[komoditas][jenis], 2)
            except ValueError:
                pass
        return np.nan

    df_final[col_urea] = df_final.apply(lambda r: hitung_pupuk(r, 'urea'), axis=1)
    df_final[col_npk] = df_final.apply(lambda r: hitung_pupuk(r, 'npk'), axis=1)
    df_final[col_npk_form] = df_final.apply(lambda r: hitung_pupuk(r, 'npk_formula'), axis=1)
    df_final[col_organik] = df_final.apply(lambda r: hitung_pupuk(r, 'organik'), axis=1)
    df_final[col_za] = df_final.apply(lambda r: hitung_pupuk(r, 'za'), axis=1)

print("Perhitungan seluruh dosis pupuk berhasil!")

Perhitungan seluruh dosis pupuk berhasil!


In [7]:
# Format NIK agar tidak berubah jadi angka ilmiah
df_final['KTP'] = df_final['KTP'].apply(lambda x: f"'{clean_nik(x)}" if clean_nik(x) != "" else "")

output_file = "template-erdkk-2027 - Sumber Makmur (Final Terhitung).xlsx"
df_final.to_excel(output_file, index=False, sheet_name="2027")

print(f"Selesai! File tersimpan sebagai: {output_file}")
df_final.head()

Selesai! File tersimpan sebagai: template-erdkk-2027 - Sumber Makmur (Final Terhitung).xlsx


,Nama Penyuluh,KTP,Nama Ibu Kandung,Kode Desa,ID Poktan,Kode Kios Pengecer,Subsektor,Komoditas Periode Tanam I,Luas Lahan (Ha) Periode Tanam I,Pupuk Urea (Kg) Periode Tanam I,...,Pupuk NPK Formula Khusus (Kg) Periode Tanam II,Pupuk Organik (Kg) Periode Tanam II,Pupuk Za (Kg) Periode Tanam II,Komoditas Periode Tanam III,Luas Lahan (Ha) Periode Tanam III,Pupuk Urea (Kg) Periode Tanam III,Pupuk NPK (Kg) Periode Tanam III,Pupuk NPK Formula Khusus (Kg) Periode Tanam III,Pupuk Organik (Kg) Periode Tanam III,Pupuk Za (Kg) Periode Tanam III
0,"Lulut Hasri Saptiwi, S.Pt",'3303071001600003,SURTINAH,3.303072e+09,170614.0,RT0000025379,TANAMAN PANGAN,JAGUNG,0.43,107.5,...,0.0,215.0,0.0,JAGUNG,0.43,107.5,129.0,0.0,215.0,0.0
1,"Lulut Hasri Saptiwi, S.Pt",'3303072211950001,SUMINAH,3.303072e+09,170614.0,RT0000025379,TANAMAN PANGAN,JAGUNG,0.17,42.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Lulut Hasri Saptiwi, S.Pt",'3303074202890005,RUMINAH,3.303072e+09,170614.0,RT0000025379,HORTIKULTURA,CABAI,0.13,0.0,...,0.0,65.0,0.0,CABAI,0.13,0.0,52.0,0.0,65.0,0.0
3,"Lulut Hasri Saptiwi, S.Pt",'3303075207890002,KAMSIAH,3.303072e+09,170614.0,RT0000025379,TANAMAN PANGAN,UBI KAYU,0.11,22.0,...,NaN,NaN,NaN,CABAI,0.11,0.0,44.0,0.0,55.0,0.0
4,"Lulut Hasri Saptiwi, S.Pt",'3303150502590001,RIWEN,3.303072e+09,170614.0,RT0000025379,TANAMAN PANGAN,JAGUNG,0.50,125.0,...,0.0,250.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [42]:
# Sel 7: Rekapitulasi Luas Lahan per Komoditas & Musim Tanam

musim_tanam_cols = [
    ('MT 1', 'Komoditas Periode Tanam I', 'Luas Lahan (Ha) Periode Tanam I'),
    ('MT 2', 'Komoditas Periode Tanam II', 'Luas Lahan (Ha) Periode Tanam II'),
    ('MT 3', 'Komoditas Periode Tanam III', 'Luas Lahan (Ha) Periode Tanam III')
]

rekap_list = []
for mt_nama, col_kom, col_luas in musim_tanam_cols:
    sub_df = df_final[[col_kom, col_luas]].dropna(subset=[col_kom, col_luas]).copy()
    sub_df[col_kom] = sub_df[col_kom].astype(str).str.strip().str.upper()
    sub_df[col_luas] = pd.to_numeric(sub_df[col_luas], errors='coerce').fillna(0)
    
    # Agregasi jumlah petani dan total luas
    grp = sub_df.groupby(col_kom).agg(
        Jumlah_Petani=(col_luas, 'count'),
        Total_Luas_Ha=(col_luas, lambda x: round(x.sum(), 3))
    ).reset_index()
    grp['Musim Tanam'] = mt_nama
    grp.rename(columns={col_kom: 'Komoditas'}, inplace=True)
    rekap_list.append(grp)

# Gabung tabel detail rekapitulasi
df_rekap_detail = pd.concat(rekap_list, ignore_index=True)
df_rekap_detail = df_rekap_detail[['Musim Tanam', 'Komoditas', 'Jumlah_Petani', 'Total_Luas_Ha']]

# Buat tabel pivot (Matriks Komoditas vs Musim Tanam)
df_pivot = df_rekap_detail.pivot(index='Komoditas', columns='Musim Tanam', values='Total_Luas_Ha').fillna(0)
df_pivot['Total Setahun (Ha)'] = df_pivot.sum(axis=1).round(3)

print("=== TABEL REKAPITULASI LUAS LAHAN (Ha) ===")
display(df_pivot)

# Simpan rekapitulasi ke file Excel yang sama di sheet berbeda
with pd.ExcelWriter("template-erdkk-2027 - Sumber Jaya (Final Terhitung).xlsx", mode='a', if_sheet_exists='replace') as writer:
    df_pivot.to_excel(writer, sheet_name="Rekap Luas Komoditas")

print("Rekapitulasi berhasil ditambahkan ke sheet 'Rekap Luas Komoditas'!")

=== TABEL REKAPITULASI LUAS LAHAN (Ha) ===


Musim Tanam,MT 1,MT 2,MT 3,Total Setahun (Ha)
Komoditas,,,,
CABAI,0.280,0.000,0.600,0.880
JAGUNG,5.081,14.069,13.668,32.818
KOPI,0.250,0.000,0.000,0.250
PADI,11.256,1.050,0.040,12.346
UBI KAYU,0.930,0.030,0.240,1.200


Rekapitulasi berhasil ditambahkan ke sheet 'Rekap Luas Komoditas'!
